# Lecture 4: Optimization

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 4, with material from Stanford CS231n 2026, Lecture 3.*

Lecture 3 gave us a **loss** $L(W)$ that measures how bad the weights are. **Optimization** means finding the weights that make the loss as small as possible:

$$W^* = \arg\min_W L(W)$$

Picture the loss as a hilly landscape, with yourself standing on it **blindfolded**. You can't see the valley; you can only feel the slope under your feet. Which way do you step?

**What you'll learn**
1. Why random search fails, and how **gradients** tell us which way is downhill.
2. **Numerical** vs. **analytic** gradients, and **gradient checking**.
3. **Gradient descent** and **stochastic gradient descent (SGD)**, and the learning rate.
4. Problems with SGD, and the fixes: **momentum**, **RMSProp**, **Adam**.
5. **Learning-rate schedules**, and a glimpse of second-order methods.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

**Setup:** the digits data and the softmax loss from Lecture 3, with a small L2 regularization term ($\lambda = 0.001$). The regularization matters here: it gives the loss a real minimum to settle into. (Without it, these digits are so easy to separate that the loss can keep shrinking forever as the weights grow.) `softmax_loss_and_gradient` returns both the loss and its gradient (we'll see where the gradient formula comes from in Lecture 6).

In [ ]:
digits = load_digits()
images = digits.images / 16.0
labels = digits.target
rng = np.random.default_rng(0)
order = rng.permutation(len(images))
train_idx, test_idx = order[:1300], order[1300:]
mean_image = images[train_idx].mean(axis=0)

def prepare(idx):
    flat = (images[idx] - mean_image).reshape(len(idx), 64)
    return np.column_stack([flat, np.ones(len(idx))])      # 64 pixels + bias

X_train, y_train = prepare(train_idx), labels[train_idx]
X_test, y_test = prepare(test_idx), labels[test_idx]

def softmax_loss_and_gradient(W, X, y, reg=0.001):
    scores = X @ W
    scores = scores - scores.max(axis=1, keepdims=True)
    probs = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)
    loss = -np.mean(np.log(probs[np.arange(len(y)), y])) + reg * np.sum(W * W)
    probs[np.arange(len(y)), y] -= 1
    gradient = X.T @ probs / len(y) + 2 * reg * W
    return loss, gradient

def accuracy(W, X, y):
    return np.mean(np.argmax(X @ W, axis=1) == y)

## 1. Idea #1: random search (a bad idea)

Try lots of random weights and keep the best:

In [ ]:
best_loss, best_W = float("inf"), None
for attempt in range(1000):
    W = rng.normal(size=(65, 10))
    loss, _ = softmax_loss_and_gradient(W, X_train, y_train)
    if loss < best_loss:
        best_loss, best_W = loss, W
print(f"best of 1000 random tries: loss {best_loss:.2f}, test accuracy {accuracy(best_W, X_test, y_test):.1%}")

Better than guessing (10%), but bad. On CIFAR-10 the slides report 15.5% this way. In 650 dimensions, a random guess almost never lands anywhere good.

## 2. Idea #2: follow the slope

**In 1-D**, the slope of a function is its derivative:

$$\frac{df}{dx} = \lim_{h\to0}\frac{f(x + h) - f(x)}{h}$$

**In many dimensions**, the **gradient** $\nabla L$ is the list of slopes along each axis (the partial derivatives). The key fact: **the gradient points uphill (steepest increase), so $-\nabla L$ points downhill.**

### Computing the gradient numerically
The definition suggests a way: nudge one weight by a tiny $h$, see how much the loss changes, divide by $h$. Do it for every weight.

In [ ]:
def numerical_gradient(W, h=1e-5):
    grad = np.zeros_like(W)
    for i in range(W.shape[0]):
        for j in range(W.shape[1]):
            W_plus = W.copy(); W_plus[i, j] += h
            W_minus = W.copy(); W_minus[i, j] -= h
            loss_plus, _ = softmax_loss_and_gradient(W_plus, X_train, y_train)
            loss_minus, _ = softmax_loss_and_gradient(W_minus, X_train, y_train)
            grad[i, j] = (loss_plus - loss_minus) / (2 * h)       # the slope along this one weight
    return grad

(We use $f(x + h) - f(x - h)$ over $2h$, the **centered difference**, which is more accurate than the one-sided formula.) Let's time it against the formula-based (**analytic**) gradient:

In [ ]:
import time
W_example = rng.normal(0, 0.1, size=(65, 10))

start = time.time()
grad_numerical = numerical_gradient(W_example)
time_numerical = time.time() - start

start = time.time()
_, grad_analytic = softmax_loss_and_gradient(W_example, X_train, y_train)
time_analytic = time.time() - start

print(f"numerical gradient: {time_numerical:.2f} seconds (2 loss computations per weight, 650 weights)")
print(f"analytic gradient:  {time_analytic * 1000:.1f} milliseconds")
print(f"largest difference between them: {np.abs(grad_numerical - grad_analytic).max():.1e}")

| | Numerical gradient | Analytic gradient |
|---|---|---|
| Speed | slow | fast |
| Accuracy | approximate | exact |
| Writing it | easy | error-prone |

**In practice: always use the analytic gradient, but check it against the numerical one.** That's called a **gradient check**, and you'll do it in every assignment. A tiny difference (like 1e-9 above) means your formula is right.

## 3. Gradient descent

Repeatedly take a small step downhill:
```python
w = initialize_weights()
for t in range(num_steps):
    dw = compute_gradient(loss_fn, data, w)
    w -= learning_rate * dw
```
The **learning rate** (step size) is the most important hyperparameter. Let's try four:

In [ ]:
def gradient_descent(learning_rate, steps=150):
    W = np.zeros((65, 10))
    losses = []
    for step in range(steps):
        loss, gradient = softmax_loss_and_gradient(W, X_train, y_train)
        losses.append(loss)
        W = W - learning_rate * gradient
    return W, losses

plt.figure(figsize=(6, 3.5))
for lr in [0.01, 0.5, 5.0, 60.0]:
    W, losses = gradient_descent(lr)
    plt.plot(losses, label=f"learning rate {lr}")
    print(f"learning rate {lr:5}: final loss {losses[-1]:.3f}, test accuracy {accuracy(W, X_test, y_test):.1%}")
plt.yscale("log"); plt.xlabel("step"); plt.ylabel("training loss"); plt.legend()
plt.show()

**What we see:** too small (0.01) crawls; a good one (0.5 to 5) converges fast; too big (60) overshoots the valley, so the loss bounces up and down instead of settling.

## 4. Stochastic gradient descent (SGD)

The full gradient is an **average over all training examples**. With millions of examples, one step costs a full pass over the data. Instead, estimate the gradient from a small random **minibatch** (32, 64, 128 examples are common):
```python
for t in range(num_steps):
    minibatch = sample_data(data, batch_size)
    dw = compute_gradient(loss_fn, minibatch, w)
    w -= learning_rate * dw
```
The estimate is **noisy**, but each step is far cheaper. Compare with the same total amount of computation (each full step uses 1,300 examples; each SGD step uses 32):

In [ ]:
W_full = np.zeros((65, 10))
for step in range(20):                          # 20 full steps = 26,000 example-gradients
    _, gradient = softmax_loss_and_gradient(W_full, X_train, y_train)
    W_full -= 0.5 * gradient

W_sgd = np.zeros((65, 10))
for step in range(812):                         # 812 minibatch steps x 32 = about 26,000 example-gradients
    batch = rng.integers(0, len(X_train), 32)
    _, gradient = softmax_loss_and_gradient(W_sgd, X_train[batch], y_train[batch])
    W_sgd -= 0.5 * gradient

print(f"full gradient descent, 20 steps:  test accuracy {accuracy(W_full, X_test, y_test):.1%}")
print(f"SGD with minibatches, 812 steps:  test accuracy {accuracy(W_sgd, X_test, y_test):.1%}")

**What we see:** for the same computation, many noisy small steps beat a few exact ones.

## 5. Problems with SGD

To see what goes wrong, let's switch to a simple 2-D "loss landscape" we can draw: a long, narrow valley,

$$L(w_1, w_2) = \tfrac{1}{2}\left(w_1^2 + 20\,w_2^2\right)$$

It's **steep** in the $w_2$ direction and **flat** in the $w_1$ direction. The minimum is at $(0, 0)$.

In [ ]:
def valley_loss(w):
    return 0.5 * (w[0]**2 + 20 * w[1]**2)

def valley_gradient(w):
    return np.array([w[0], 20 * w[1]])

start_point = np.array([-9.0, 1.5])

def draw_valley():
    a, b = np.meshgrid(np.linspace(-10, 2, 200), np.linspace(-2, 2, 200))
    plt.contour(a, b, 0.5 * (a**2 + 20 * b**2), levels=20, colors="lightgray")

In [ ]:
def run_sgd(learning_rate, steps=40):
    w = start_point.copy()
    path = [w.copy()]
    for step in range(steps):
        w = w - learning_rate * valley_gradient(w)
        path.append(w.copy())
    return np.array(path)

path = run_sgd(0.09)
plt.figure(figsize=(7, 3)); draw_valley()
plt.plot(path[:, 0], path[:, 1], "o-", markersize=3)
plt.title(f"gradient descent: loss after 40 steps = {valley_loss(path[-1]):.3f}")
plt.show()

**Problem 1: poor conditioning.** The path **zig-zags** across the steep direction while crawling along the flat one. We can't just raise the learning rate: the steep direction would blow up. (The ratio of steepest to flattest curvature, 20 here, is the **condition number**.)

**Problem 2: local minima and saddle points.** Where the gradient is zero, gradient descent stops. In high dimensions, **saddle points** (uphill in some directions, downhill in others) are far more common than local minima, and the gradient near them is tiny, so progress slows down.

**Problem 3: noise.** Minibatch gradients are noisy, so the path wanders.

## 6. SGD + Momentum

**Keep moving in the direction you've been going.** Build up a **velocity** as a running average of gradients; $\rho$ (typically 0.9) acts like friction:

$$v_{t+1} = \rho\,v_t + \nabla L(w_t), \qquad w_{t+1} = w_t - \alpha\,v_{t+1}$$

The zig-zags cancel out in the average, while the consistent push along the valley adds up.

In [ ]:
def run_momentum(learning_rate, rho=0.8, steps=40):
    w = start_point.copy()
    velocity = np.zeros(2)
    path = [w.copy()]
    for step in range(steps):
        velocity = rho * velocity + valley_gradient(w)
        w = w - learning_rate * velocity
        path.append(w.copy())
    return np.array(path)

path_sgd = run_sgd(0.09)
path_momentum = run_momentum(0.02)
plt.figure(figsize=(7, 3)); draw_valley()
plt.plot(path_sgd[:, 0], path_sgd[:, 1], "o-", markersize=3, label=f"SGD: final loss {valley_loss(path_sgd[-1]):.4f}")
plt.plot(path_momentum[:, 0], path_momentum[:, 1], "s-", markersize=3, label=f"momentum: final loss {valley_loss(path_momentum[-1]):.4f}")
plt.legend(fontsize=8); plt.show()

**Nesterov momentum** is a small improvement: compute the gradient at the point where the velocity is about to take you (a "look-ahead"), then correct. It overshoots less.

## 7. Adaptive methods: AdaGrad, RMSProp, Adam

Another fix for poor conditioning: give **each weight its own step size**. Divide each step by the size of that weight's recent gradients: steep directions (big gradients) get smaller steps, flat directions get bigger ones.

**AdaGrad** (2011) adds up all past squared gradients:
```python
grad_squared += dw * dw
w -= learning_rate * dw / (np.sqrt(grad_squared) + 1e-7)
```
Problem: `grad_squared` only grows, so the steps eventually shrink to zero.

**RMSProp** (2012) uses a *decaying* average instead, so old gradients are forgotten:
```python
grad_squared = decay * grad_squared + (1 - decay) * dw * dw
w -= learning_rate * dw / (np.sqrt(grad_squared) + 1e-7)
```

**Adam** (2015) = momentum + RMSProp + a "bias correction":

In [ ]:
def run_adam(learning_rate, steps=40, beta1=0.9, beta2=0.999):
    w = start_point.copy()
    m = np.zeros(2)          # running average of gradients (momentum)
    v = np.zeros(2)          # running average of squared gradients (RMSProp)
    path = [w.copy()]
    for t in range(1, steps + 1):
        g = valley_gradient(w)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g * g
        m_corrected = m / (1 - beta1**t)          # bias correction (see below)
        v_corrected = v / (1 - beta2**t)
        w = w - learning_rate * m_corrected / (np.sqrt(v_corrected) + 1e-7)
        path.append(w.copy())
    return np.array(path)

path_adam = run_adam(1.0)
plt.figure(figsize=(7, 3)); draw_valley()
plt.plot(path_sgd[:, 0], path_sgd[:, 1], "o-", markersize=3, label="SGD")
plt.plot(path_adam[:, 0], path_adam[:, 1], "s-", markersize=3, label="Adam")
plt.legend(fontsize=8); plt.title("Adam rescales each direction, so it heads more directly to the minimum"); plt.show()

**Why bias correction?** `m` and `v` start at 0, so after the first step they're much too small (especially `v`, since `beta2 = 0.999` means it barely moves). Without correction, the first step would be about $0.1/\sqrt{0.001}\approx3.2$ times too big. Dividing by $(1 - \beta^t)$ fixes that:

In [ ]:
g = 0.5                                     # pretend the gradient is 0.5 on the first step
m, v = 0.1 * g, 0.001 * g * g               # moments after one step, starting from 0
print("first step without correction:", round(m / np.sqrt(v), 3), "x the learning rate")
print("first step with correction:   ", round((m / 0.1) / np.sqrt(v / 0.001), 3), "x the learning rate")

**Good default:** Adam with `beta1 = 0.9`, `beta2 = 0.999`, and learning rate `1e-3` or `5e-4` works for many models.

**AdamW** (Loshchilov & Hutter, 2019): with Adam, adding L2 regularization to the loss is *not* the same as shrinking the weights directly ("weight decay"), because the L2 gradient gets rescaled along with everything else. AdamW applies the weight decay separately, after the Adam step. It's the standard choice for training Transformers.

## 8. Comparing them on the digits

Minibatch training of our softmax classifier with each method (learning rates chosen to suit each one):

In [ ]:
def train_digits(method, learning_rate, steps=300, batch_size=64):
    rng_local = np.random.default_rng(0)
    W = np.zeros((65, 10))
    velocity, m, v = np.zeros_like(W), np.zeros_like(W), np.zeros_like(W)
    for t in range(1, steps + 1):
        batch = rng_local.integers(0, len(X_train), batch_size)
        _, g = softmax_loss_and_gradient(W, X_train[batch], y_train[batch])
        if method == "sgd":
            W -= learning_rate * g
        elif method == "momentum":
            velocity = 0.9 * velocity + g
            W -= learning_rate * velocity
        elif method == "adam":
            m = 0.9 * m + 0.1 * g
            v = 0.999 * v + 0.001 * g * g
            W -= learning_rate * (m / (1 - 0.9**t)) / (np.sqrt(v / (1 - 0.999**t)) + 1e-7)
    final_loss, _ = softmax_loss_and_gradient(W, X_train, y_train)
    return final_loss, accuracy(W, X_test, y_test)

for method, lr in [("sgd", 0.5), ("momentum", 0.05), ("adam", 0.01)]:
    loss, acc = train_digits(method, lr)
    print(f"{method:9s}: final training loss {loss:.3f}, test accuracy {acc:.1%}")

On this easy problem all three end up in a similar place once their learning rates are reasonable. The differences grow on harder, badly conditioned problems like deep networks.

## 9. Learning-rate schedules

**Start with a large learning rate, then make it smaller over time.** Large steps make fast progress early; small steps let the weights settle into the valley at the end. Common schedules ($\alpha_0$ = initial rate, $T$ = total epochs):
* **Step:** multiply by 0.1 at a few fixed epochs (e.g., ResNets: epochs 30, 60, 90).
* **Cosine:** $\alpha_t = \tfrac12\alpha_0\left(1 + \cos(\pi t/T)\right)$
* **Linear:** $\alpha_t = \alpha_0(1 - t/T)$
* **Inverse square root:** $\alpha_t = \alpha_0/\sqrt{t}$
* **Linear warmup:** start from 0 and increase over the first few thousand steps (prevents early blow-ups).

In [ ]:
T = 100
epochs = np.arange(T)
step_schedule = [0.1 ** sum(t >= m for m in (30, 60, 90)) for t in epochs]
cosine_schedule = 0.5 * (1 + np.cos(np.pi * epochs / T))
linear_schedule = 1 - epochs / T

plt.figure(figsize=(6, 3))
plt.plot(epochs, step_schedule, label="step")
plt.plot(epochs, cosine_schedule, label="cosine")
plt.plot(epochs, linear_schedule, label="linear")
plt.xlabel("epoch"); plt.ylabel("learning rate (fraction of initial)"); plt.legend()
plt.show()

Does it help? Train with a large learning rate and small minibatches (very noisy steps), with and without a cosine schedule:

In [ ]:
def train_with_schedule(use_cosine, initial_lr=5.0, epochs=40, batch_size=8):
    rng_local = np.random.default_rng(0)
    W = np.zeros((65, 10))
    for epoch in range(epochs):
        lr = initial_lr * 0.5 * (1 + np.cos(np.pi * epoch / epochs)) if use_cosine else initial_lr
        order_local = rng_local.permutation(len(X_train))
        for start in range(0, len(X_train), batch_size):
            batch = order_local[start:start + batch_size]
            _, g = softmax_loss_and_gradient(W, X_train[batch], y_train[batch])
            W -= lr * g
    loss, _ = softmax_loss_and_gradient(W, X_train, y_train)
    return loss

print(f"constant learning rate: final training loss {train_with_schedule(False):.4f}")
print(f"cosine schedule:        final training loss {train_with_schedule(True):.4f}")

**What we see:** with a constant large rate, the noisy steps keep bouncing around the minimum; decaying the rate lets them settle, so the final loss is lower.

## 10. Second-order optimization (a glimpse)

Gradient descent uses only the **slope**. Second-order methods also use the **curvature** (the **Hessian**, the matrix of second derivatives) and jump straight to the bottom of the local bowl. **Newton's method:**

$$w^* = w_0 - H^{-1}\nabla L(w_0)$$

On our valley it lands on the minimum **in one step**, whatever the conditioning:

In [ ]:
H = np.array([[1, 0], [0, 20]])                       # the valley's curvature (second derivatives)
w0 = start_point
w1 = w0 - np.linalg.inv(H) @ valley_gradient(w0)
print("start:", w0, "-> after one Newton step:", w1)

**Why don't we use it for deep learning?** The Hessian has $N^2$ entries and inverting it costs about $N^3$ operations. With $N$ = 100 million weights, that's impossible. **L-BFGS** approximates it cheaply and works well for full-batch problems, but not with noisy minibatches.

**In practice:**
* **Adam (or AdamW) is a good default.**
* **SGD + momentum** can do better, but needs more tuning.
* If you can use the full dataset every step (no noise), try **L-BFGS**.

## Summary
* The **gradient** points uphill; step the other way: $w \leftarrow w - \alpha\nabla L$.
* Use the **analytic** gradient; check it with the **numerical** one.
* **SGD** uses minibatches: noisy but cheap steps.
* The **learning rate** matters most: too small is slow, too big diverges.
* **Momentum** smooths zig-zags; **RMSProp/Adam** give each weight its own step size.
* **Decay the learning rate** over time.

### Check your understanding
1. Why does the negative gradient point downhill?
2. Why is SGD usually better than full gradient descent for the same amount of computation?
3. Why can't we just use a bigger learning rate in a narrow valley?
4. What problem does Adam's bias correction fix?

<details><summary>Answers</summary>

1. The gradient points in the direction of steepest increase, so its opposite is the direction of steepest decrease.
2. Each step is far cheaper, so you take many more steps; the noise averages out over steps.
3. The steep direction would overshoot and blow up; the learning rate is limited by the steepest direction.
4. The running averages start at 0, which makes the first steps too large; dividing by $1 - \beta^t$ removes that bias.
</details>

**Next:** Lecture 5, neural networks.